# 38. Daily crashes in international equity indices: preregistered historical external validation

Here I apply the measurement framework of the Bitcoin manuscript (AUC decomposition, pre-onset feasibility F, model choice depending on the definition, event-level catches under a budget) unchanged to another asset at another frequency, and I evaluate it once.
The data are daily closes of 10 major equity indices from Yahoo Finance. I train M0 and GBM on data up to 1999, calibrate thresholds on 2000–2004 without outcome labels, and evaluate on 2005–2025.
I preregistered this one-shot backcast as a historical external validation on OSF (DOI 10.17605/OSF.IO/BKJUQ); because 2005–2025 has already happened, it is not a prospective test.
I registered the hashes of the analysis specification (`SPEC`) and the core code (`38_core.py`) and downloaded the data only after registration; in phase 1 I freeze the models and thresholds, and in phase 2 I compute the evaluation results once, for the first time.
I write outputs to `external_equity_indices_v1`; they contain no index levels (dates, positions, counts and statistics only).
I record the pre-specified readings (Q1–Q4) in `38_SPEC.json`.

In [1]:
from pathlib import Path
import os, sys, json, time, hashlib, importlib.util, subprocess
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------------------------------------------------------- run parameters (not part of the SPEC)
REGISTRATION_DOI = '10.17605/OSF.IO/BKJUQ'
REQUIRE_REGISTRATION = True      # refuse to download data before a registration DOI is entered
LOCAL_RAW = os.environ.get('NB38_LOCAL_RAW')            # synthetic-data test only
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive') / 'btc_lead_cost_results_77months_spot'
except ImportError:
    BASE = Path(os.environ.get('NB38_BASE', '.'))
OUT = BASE / ('external_equity_indices_v1' + ('_SYNTHETIC_TEST' if LOCAL_RAW else ''))
OUT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


### 1a. Frozen core code (`38_core.py`)

In [2]:
CORE_SRC = r'''"""Notebook 38 core: daily equity-index crashes, a pre-registered historical external validation.

Every function the analysis uses is in this file; the notebook only calls them. Its SHA-256 is part of
the frozen specification (38_SPEC.json), so the code that produced the results is the code that was registered.
No function writes index levels to any output: outputs carry dates, positions, counts and statistics only.
"""
import io
import json
import time
import hashlib
import urllib.request
import numpy as np
import pandas as pd

CORE_VERSION = '38-core-1'


# ----------------------------------------------------------------------------- data
def _yahoo(ticker, start, end):
    import yfinance as yf
    df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False, threads=False)
    if df is None or len(df) == 0:
        return None
    col = df['Close']
    if isinstance(col, pd.DataFrame):
        col = col.iloc[:, 0]
    s = pd.Series(col.to_numpy(float), index=pd.to_datetime(df.index).tz_localize(None).normalize())
    return s


def _stooq(symbol, start, end):
    url = f'https://stooq.com/q/d/l/?s={symbol}&i=d'
    with urllib.request.urlopen(url, timeout=60) as r:
        raw = r.read().decode('utf-8', 'replace')
    if not raw.startswith('Date'):
        return None
    df = pd.read_csv(io.StringIO(raw))
    s = pd.Series(df['Close'].to_numpy(float), index=pd.to_datetime(df['Date']))
    return s[(s.index >= pd.Timestamp(start)) & (s.index < pd.Timestamp(end))]


def clean(s):
    """Drop missing or non-positive closes and duplicate dates (keep the last); sort by date."""
    s = pd.Series(s).dropna()
    s = s[s > 0]
    s = s[~s.index.duplicated(keep='last')].sort_index()
    s.index.name = 'date'
    return s.astype(float)


def coverage_ok(s, spec):
    pre = int((s.index < pd.Timestamp(spec['periods']['calibration'][0])).sum())
    ev = int(((s.index >= pd.Timestamp(spec['periods']['evaluation'][0])) & (s.index < pd.Timestamp(spec['periods']['evaluation'][1]))).sum())
    return pre >= spec['data']['min_days_before_calibration'] and ev >= spec['data']['min_days_in_evaluation'], pre, ev


def fetch_index(ticker, spec, log=print):
    """Yahoo first; if it fails or does not cover, Stooq for the whole series; never mixed."""
    start, end = spec['data']['download_start'], spec['data']['download_end']
    tried = []
    for source in ['yahoo', 'stooq']:
        try:
            if source == 'yahoo':
                s = _yahoo(ticker, start, end)
            else:
                sym = spec['data']['stooq_symbols'].get(ticker)
                s = _stooq(sym, start, end) if sym else None
        except Exception as e:  # network or parser failure
            tried.append(f'{source}: {type(e).__name__}: {str(e)[:80]}'); s = None
        if s is None or len(s) == 0:
            tried.append(f'{source}: no data'); continue
        s = clean(s)
        ok, pre, ev = coverage_ok(s, spec)
        if ok:
            return s, source, tried
        tried.append(f'{source}: coverage {pre} days before calibration, {ev} in evaluation')
    return None, None, tried


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


# ----------------------------------------------------------------------------- features
FEATS = ['z_vol', 'ret_1', 'ret_5', 'ret_20', 'ret_60', 'rv_5', 'rv_20', 'rv_60', 'rv_ratio_5_60',
         'dd_20', 'dd_60', 'dd_250', 'dist_ma200']


def features(close):
    """Causal daily features (data up to the close of day t)."""
    c = close.astype(float); lc = np.log(c); r = lc.diff()
    F = pd.DataFrame(index=close.index)
    for k in (5, 20, 60):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k).std()
    lrv = np.log(F['rv_20'].where(F['rv_20'] > 0))
    base = lrv.rolling(250, min_periods=125)
    F['z_vol'] = (lrv - base.mean().shift(1)) / base.std().shift(1).replace(0, np.nan)
    for k in (1, 5, 20, 60):
        F[f'ret_{k}'] = lc - lc.shift(k)
    F['rv_ratio_5_60'] = F['rv_5'] / F['rv_60']
    for k in (20, 60, 250):
        F[f'dd_{k}'] = 1 - c / c.rolling(k, min_periods=k).max()
    F['dist_ma200'] = lc - np.log(c.rolling(200, min_periods=200).mean())
    F = F.replace([np.inf, -np.inf], np.nan)
    F['B0'] = F['rv_20']
    F['B1'] = F['dd_20']
    return F


# ----------------------------------------------------------------------------- events
def catalog(close, H, theta=0.10, refractory=20, onset_frac=0.25):
    """Breach: close <= (1 - theta) x max of the previous H closes (first day of a run; sustain 1).
    Peak: that maximum (first occurrence). Onset: first close after the peak <= (1 - onset_frac x theta) x peak.
    Next event: at least `refractory` trading days after the previous breach and a peak after it."""
    c = np.asarray(close, float); n = len(c)
    prior = pd.Series(c).rolling(H, min_periods=H).max().shift(1).to_numpy()
    raw = np.isfinite(prior) & (c <= (1 - theta) * prior)
    fresh = raw & ~np.r_[False, raw[:-1]]
    rows = []; last = -10**9
    for i in np.flatnonzero(fresh):
        if i - H < 0 or i - last < refractory:
            continue
        peak = int(i - H + np.argmax(c[i - H:i]))
        if peak <= last:
            continue
        cand = np.flatnonzero(c[peak + 1:i + 1] <= (1 - onset_frac * theta) * c[peak])
        onset = int(peak + 1 + cand[0])
        rows.append(dict(peak_pos=peak, onset_pos=onset, breach_pos=int(i)))
        last = i
    E = pd.DataFrame(rows, columns=['peak_pos', 'onset_pos', 'breach_pos'])
    idx = pd.DatetimeIndex(close.index)
    for k in ['peak', 'onset', 'breach']:
        E[f'{k}_date'] = idx[E[f'{k}_pos'].to_numpy()] if len(E) else pd.Series([], dtype='datetime64[ns]')
    E['onset_to_breach_days'] = E.breach_pos - E.onset_pos
    E['peak_to_breach_days'] = E.breach_pos - E.peak_pos
    return E


def window_labels(n, E, wmin=1, wmax=20):
    """Per-day labels: 0 negative, 1 pre-onset positive, 2 in-progress positive; plus the event index of each positive."""
    lab = np.zeros(n, np.int8); ev = np.full(n, -1, np.int64)
    for k, e in enumerate(E.itertuples(index=False)):
        lo, hi = e.breach_pos - wmax, e.breach_pos - wmin
        for t in range(max(lo, 0), min(hi, n - 1) + 1):
            lab[t] = 1 if t < e.onset_pos else 2; ev[t] = k
    return lab, ev


def feasibility(E, wmax, wmin=1):
    """Event e can be warned before onset iff some day a has wmin <= b - a <= wmax and a < onset."""
    L = np.minimum(E.breach_pos - wmin, E.onset_pos - 1) - (E.breach_pos - wmax) + 1
    return np.maximum(L, 0).to_numpy()


# ----------------------------------------------------------------------------- scheduler and calibration
def schedule(pos, dates, eligible, q_cap=1, y_cap=4, cooldown=20):
    """Calendar scheduler on trading-day positions: at most q_cap per calendar quarter, y_cap per calendar year,
    and at least `cooldown` trading days between alerts. Returns alert positions."""
    pos = np.asarray(pos, np.int64); el = np.asarray(eligible, bool)
    d = pd.DatetimeIndex(dates)
    yr = d.year.to_numpy(); qt = yr * 4 + (d.month.to_numpy() - 1) // 3
    out = []; last = -10**9; qc = {}; yc = {}
    for k in np.flatnonzero(el):
        p = int(pos[k])
        if p - last < cooldown or qc.get(qt[k], 0) >= q_cap or yc.get(yr[k], 0) >= y_cap:
            continue
        out.append(p); last = p; qc[qt[k]] = qc.get(qt[k], 0) + 1; yc[yr[k]] = yc.get(yr[k], 0) + 1
    return np.asarray(out, np.int64)


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate(pos, dates, scores, sched):
    """Crypto notebook-5 rule: the most alerts the budget allows on the calibration days, then the strictest threshold."""
    rows = []
    for th in score_grid(scores):
        e = scores >= th
        rows.append((float(th), len(schedule(pos, dates, e, **sched)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts)


# ----------------------------------------------------------------------------- matching
def match(E, alerts, wmin=1, wmax=20):
    """Per event: first alert with wmin <= b - a <= wmax (windows of different events do not overlap)."""
    a = np.sort(np.asarray(alerts, np.int64)); b = E.breach_pos.to_numpy(np.int64); o = E.onset_pos.to_numpy(np.int64)
    if len(a) == 0:
        z = np.zeros(len(b), bool); return z, z.copy(), np.full(len(b), -1, np.int64)
    lo = np.searchsorted(a, b - wmax, 'left'); hi = np.searchsorted(a, b - wmin, 'right')
    hit = hi > lo
    al = np.where(hit, a[np.minimum(lo, len(a) - 1)], -1)
    pre = hit & (al < o)
    return hit, pre, al


# ----------------------------------------------------------------------------- AUC
def auc_weighted(s, pos, w):
    """Mann-Whitney AUC of positives (pos True) over negatives with day weights w (ties count one half)."""
    s = np.asarray(s, float); pos = np.asarray(pos, bool); w = np.asarray(w, float)
    o = np.argsort(s, kind='mergesort'); s, pos, w = s[o], pos[o], w[o]
    _, g = np.unique(s, return_inverse=True)
    wp = np.bincount(g, w * pos); wn = np.bincount(g, w * ~pos)
    Wp, Wn = wp.sum(), wn.sum()
    if Wp <= 0 or Wn <= 0:
        return np.nan
    below = np.cumsum(wn) - wn
    return float((wp * (below + 0.5 * wn)).sum() / (Wp * Wn))


class AucPrep:
    """Pre-sorted structure for repeated weighted AUCs of one score on one index (positives vs negatives)."""

    def __init__(self, s, pos, keep, group):
        k = np.flatnonzero(keep)
        o = k[np.argsort(s[k], kind='mergesort')]
        self.pos = pos[o].astype(bool); self.group = group[o]
        _, self.g = np.unique(s[o], return_inverse=True); self.G = int(self.g.max()) + 1 if len(o) else 0

    def auc(self, wday):
        w = wday[self.group]
        wp = np.bincount(self.g, w * self.pos, minlength=self.G); wn = np.bincount(self.g, w * ~self.pos, minlength=self.G)
        Wp, Wn = wp.sum(), wn.sum()
        if Wp <= 0 or Wn <= 0:
            return np.nan
        return float((wp * (np.cumsum(wn) - wn + 0.5 * wn)).sum() / (Wp * Wn))


class StratAucPrep:
    """Stratified (within-decile) AUC: pairs are compared only inside the same stratum."""

    def __init__(self, s, pos, keep, group, strat):
        self.parts = []
        for z in np.unique(strat[keep]):
            kz = keep & (strat == z)
            if (pos & kz).any() and (~pos & kz).any():
                self.parts.append(AucPrep(s, pos, kz, group))

    def auc(self, wday):
        num = den = 0.0
        for p in self.parts:
            w = wday[p.group]
            wp = np.bincount(p.g, w * p.pos, minlength=p.G); wn = np.bincount(p.g, w * ~p.pos, minlength=p.G)
            num += float((wp * (np.cumsum(wn) - wn + 0.5 * wn)).sum()); den += float(wp.sum() * wn.sum())
        return num / den if den > 0 else np.nan


# ----------------------------------------------------------------------------- nulls
def quarter_positions(eval_pos, quarter_of):
    """Map each calendar quarter to the evaluation positions in it (computed once per index)."""
    q = quarter_of[eval_pos]
    return {int(z): eval_pos[q == z] for z in np.unique(q)}


def null_budget_preserving(rng, alerts, qpos, quarter_of, decile_of, cooldown=20, stats=None):
    """Re-place each alert, in random order, within its own calendar quarter, at least `cooldown` days from its
    original position and from every alert already placed; own RV decile first, then other deciles by distance,
    then any eligible day of the quarter; keep the original only if nothing qualifies (notebook-26 logic, daily)."""
    a = np.asarray(alerts, np.int64); out = np.empty_like(a); placed = []
    for k in rng.permutation(len(a)):
        p0 = a[k]; q = int(quarter_of[p0]); d0 = decile_of[p0]
        cand = qpos[q]
        cand = cand[np.abs(cand - p0) >= cooldown]
        if placed:
            P = np.asarray(placed)
            cand = cand[np.all(np.abs(cand[:, None] - P[None, :]) >= cooldown, axis=1)]
        choice = p0; how = 'kept original'
        if len(cand):
            dc = decile_of[cand]
            for dist in range(10):
                pool = cand[np.abs(dc - d0) == dist]
                if len(pool):
                    choice = int(pool[rng.integers(len(pool))]); how = 'own decile' if dist == 0 else 'other decile'; break
        if stats is not None:
            stats[how] = stats.get(how, 0) + 1
        out[k] = choice; placed.append(choice)
    return np.sort(out)


def null_shift(alerts_by_index, dates_by_index, eval_span, delta_days):
    """Shift every index's alert dates by the same number of calendar days, circularly within the evaluation span,
    and snap each to the first trading day of that index on or after the shifted date (wrapping to the start)."""
    t0, t1 = (pd.Timestamp(x) for x in eval_span); span = (t1 - t0).days
    out = {}
    for s, a in alerts_by_index.items():
        idx = dates_by_index[s]; ev = np.flatnonzero((idx >= t0) & (idx < t1))
        d = idx[a]
        off = ((d - t0).days.to_numpy() + int(delta_days)) % span
        nd = t0 + pd.to_timedelta(off, unit='D')
        j = np.searchsorted(idx[ev].to_numpy(), nd.to_numpy(), 'left')
        j = np.where(j >= len(ev), 0, j)
        out[s] = np.sort(ev[j])
    return out


def holm(ps):
    ps = np.asarray(ps, float); order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * ps[i])); adj[i] = run
    return adj


def episodes(breach_dates, gap_days):
    """Single linkage of breach dates across indices: a new episode starts when the gap exceeds gap_days."""
    d = pd.DatetimeIndex(breach_dates); o = np.argsort(d.values, kind='stable')
    gaps = np.r_[np.inf, np.diff(d.values[o]).astype('timedelta64[D]').astype(float)]
    ep = np.empty(len(d), np.int64); ep[o] = np.cumsum(gaps > gap_days) - 1
    return ep
'''
CORE_SHA = hashlib.sha256(CORE_SRC.encode('utf-8')).hexdigest()
(OUT / '38_core.py').write_text(CORE_SRC, encoding='utf-8')
_spec = importlib.util.spec_from_file_location('core38', OUT / '38_core.py'); C = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(C)

### 1b. Analysis specification (SPEC)

In [3]:
SPEC = {
    'title': 'Pre-registered historical external validation of onset-aware crash-warning evaluation across international equity indices',
    'notebook': '38_External_Equity_Indices', 'status': 'pre-specified one-shot backcast (historical, not prospective)',
    'prior_contact': 'no model was run on these series before this specification was frozen; notebook 30 used daily S&P 500 labels only '
                     '(Appendix C.2 of the Bitcoin manuscript); general knowledge of market history is acknowledged',
    'registration': 'OSF Registries, a separate registration linked to 10.17605/OSF.IO/TB245; data are downloaded only after it',
    'core_sha256': CORE_SHA, 'core_version': C.CORE_VERSION,
    'data': {
        'indices': ['^GSPC', '^IXIC', '^GSPTSE', '^FTSE', '^GDAXI', '^FCHI', '^SSMI', '^N225', '^HSI', '^AXJO'],
        'source_order': ['yahoo (yfinance, Close, auto_adjust=False)', 'stooq (whole series; never mixed with yahoo)'],
        'stooq_symbols': {'^GSPC': '^spx', '^IXIC': '^ndq', '^GSPTSE': '^tsx', '^FTSE': '^ukx', '^GDAXI': '^dax',
                          '^FCHI': '^cac', '^SSMI': '^smi', '^N225': '^nkx', '^HSI': '^hsi'},
        'download_start': '1950-01-01', 'download_end': '2026-01-01',
        'cleaning': 'drop missing or non-positive closes and duplicate dates (keep last); nothing else',
        'min_days_before_calibration': 1000, 'min_days_in_evaluation': 4500, 'min_indices': 6,
        'outputs_carry_no_index_levels': True},
    'periods': {'training': ['first valid feature day', '1999-12-31'], 'training_drop_last_days': 25,
                'calibration': ['2000-01-01', '2005-01-01'], 'evaluation': ['2005-01-01', '2026-01-01'],
                'event_must_breach_after_eval_start_days': 20},
    'events': {'theta': 0.10, 'onset_fraction_of_theta': 0.25, 'sustain_days': 1, 'refractory_days': 20,
               'new_peak_after_previous_breach': True, 'main_H': 20, 'secondary_H': [10, 60]},
    'window': {'wmin': 1, 'wmax': 20, 'alert_time': 'close of day a; credits breaches on days a+1 ... a+20',
               'pre_onset': 'a < onset day', 'negatives': 'all other evaluation days with valid scores',
               'secondary_negatives': 'also exclude days b ... b+20 after each breach'},
    'systems': {
        'B0': 'rv_20 (no-forecast volatility)', 'B1': 'dd_20 (no-forecast drawdown)', 'M1': 'z_vol (volatility threshold)',
        'M0': {'features': ['z_vol', 'ret_5'], 'model': 'StandardScaler + LogisticRegression(C=0.2, class_weight=balanced, max_iter=400, lbfgs, random_state=17)'},
        'GBM': {'features': C.FEATS, 'params': dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=50,
                                                    l2_regularization=1.0, class_weight='balanced', early_stopping=False, random_state=17)},
        'training': 'pooled over indices; label = day lies in a D20 judging window (pre-onset and in-progress both positive); '
                    'catalog built only from data before 2000-01-01; last 25 trading days before 2000 dropped',
        'missing': 'a day is scored only if every feature is finite; the same days are used for every system'},
    'budget': {'quarter_cap': 1, 'year_cap': 4, 'year_cap_note': 'never binds given the quarter cap (stated, not tuned)',
               'cooldown_days': 20, 'threshold_rule': 'per index and system on calibration days: most alerts allowed, then strictest',
               'scheduler_starts': '2005-01-01 (empty state)', 'alerting_systems': ['M0', 'M1', 'GBM']},
    'per_day': {'average': 'equal-weighted mean over indices of within-index AUCs',
                'S_in': 'mean_i[pi_i (AUC_in,i - 1/2)] / mean_i[AUC_i - 1/2]',
                'within_regime': 'pairs within the same index x rv_20-decile stratum (deciles over each index evaluation days)',
                'bootstrap': {'unit': 'calendar quarter, common to all indices', 'draws': 2000, 'seed': 20261104},
                'pooled_over_indices': 'secondary'},
    'feasibility': {'H': [10, 20, 60], 'wmax': [5, 10, 20, 40, 60], 'check': 'F = 1 whenever H <= wmax (stop if not)'},
    'event_level': {
        'definition': 'D20', 'systems': ['M0', 'M1', 'GBM'],
        'episodes': {'link_gap_calendar_days': 28, 'secondary_gap_calendar_days': 7},
        'primary_statistic': 'number of market episodes with at least one catch (pre-onset; in-progress)',
        'nulls': {'budget_preserving_regime': {'draws': 5000, 'role': 'primary',
                                               'rule': 'alerts re-placed in random order within their own calendar quarter on evaluation days, >= 20 trading days '
                                                       'from their own original and from every alert already placed; own rv_20 decile first, then the nearest '
                                                       'deciles, original kept only if nothing qualifies; shares of each placement type reported'},
                  'joint_circular': {'draws': 5000, 'shift_calendar_days': [365, 'span - 365'], 'role': 'secondary'},
                  'short_joint_shift': {'draws': 1000, 'shift_calendar_days': '+/- U[28, 84]', 'role': 'secondary'}},
        'p_value': '(1 + #(null >= observed)) / (draws + 1), one-sided',
        'effect': 'excess per 100 alerts = 100 x sum_e (o_e - p_e) / alerts, p_e from the primary null',
        'episode_bootstrap': {'draws': 10000, 'seed': 20261105}, 'seed_nulls': 20261106},
    'readings': {
        'Q1 (primary)': "M0, D20: supported if the 95% interval of S_in lies above 1/4, contradicted if below 1/4, inconclusive otherwise; "
                        "not evaluable (inconclusive) if mean_i[AUC_i - 1/2] <= 0.02 in more than 2.5% of bootstrap draws; S_in > 1/2 reported descriptively",
        'Q2 (key, event level)': "H1 (in-progress) and H2 (pre-onset): episode counts above the primary null, Holm-adjusted one-sided p < 0.05 across M0, M1, GBM. "
                                 "Supported if H1 holds for >= 2 systems and, for each of them, the episode-bootstrap 95% interval of (in-progress excess - pre-onset excess) "
                                 "per 100 alerts lies above 0; contradicted if that interval lies below 0 for >= 2 systems; inconclusive otherwise",
        'Q3 (diagnostic)': 'F for D60 at wmax = 20 reported; F < 1 expected by construction; F = 1 for D10 and D20 is a check',
        'Q4 (secondary)': 'D60, GBM - M0 on averaged standard and pre-onset AUC: supported if both 95% intervals exclude 0 on opposite sides, '
                          'contradicted if both exclude 0 on the same side, inconclusive otherwise; no direction predicted',
        'reporting': 'all outputs reported as observed, whatever they show'},
}
SPEC_BYTES = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode('utf-8')
SPEC_SHA = hashlib.sha256(SPEC_BYTES).hexdigest()
EXPECTED_SPEC_SHA256 = '994ec994626d023acf90241d3c90c3d4baaa78c930e2cea0a22c00571f3e5481'
if EXPECTED_SPEC_SHA256.startswith('@@'):
    print('(development copy: no expected hash embedded)')
elif SPEC_SHA != EXPECTED_SPEC_SHA256:
    raise RuntimeError(f'SPEC or core code differs from the registered version: {SPEC_SHA} != {EXPECTED_SPEC_SHA256}')
p_spec = OUT / '38_SPEC.json'
if p_spec.exists() and p_spec.read_bytes() != SPEC_BYTES:
    raise RuntimeError('a different 38_SPEC.json already exists in the output folder')
p_spec.write_bytes(SPEC_BYTES)
print('SPEC sha256:', SPEC_SHA, '| core sha256:', CORE_SHA)
RUN_LOG = OUT / 'RUN_LOG.json'
log = json.loads(RUN_LOG.read_text()) if RUN_LOG.exists() else []
log.append(dict(utc=pd.Timestamp.now(tz='UTC').isoformat(), event='cell 1', spec_sha256=SPEC_SHA, registration_doi=REGISTRATION_DOI))
RUN_LOG.write_text(json.dumps(log, indent=2))

SPEC sha256: 994ec994626d023acf90241d3c90c3d4baaa78c930e2cea0a22c00571f3e5481 | core sha256: 273f0f7df0f5ac939db8015a211a94272eb75c5c107b9ebb8795ee465dab04ea


216

## 2. Data download (after registration)

In [4]:
if REQUIRE_REGISTRATION and not REGISTRATION_DOI and not LOCAL_RAW:
    raise RuntimeError('Enter REGISTRATION_DOI (or set REQUIRE_REGISTRATION = False deliberately) before downloading data.')
if not LOCAL_RAW:
    try:
        import yfinance  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'yfinance'])
RAW = OUT / 'raw'; RAW.mkdir(exist_ok=True)
MAN = OUT / 'raw_manifest.json'
if MAN.exists():
    manifest = json.loads(MAN.read_text())
    for tk, m in manifest['indices'].items():
        if m.get('file') and C.sha256_bytes((RAW / m['file']).read_bytes()) != m['sha256']:
            raise RuntimeError(f'GUARD FAIL: raw file of {tk} changed since download')
    print('raw data already downloaded', manifest['downloaded_utc'], '- verified by hash')
else:
    manifest = dict(downloaded_utc=pd.Timestamp.now(tz='UTC').isoformat(), registration_doi=REGISTRATION_DOI, indices={})
    for tk in SPEC['data']['indices']:
        if LOCAL_RAW:
            f = Path(LOCAL_RAW) / f"{tk.replace('^', '')}.csv"
            s = C.clean(pd.read_csv(f, index_col=0, parse_dates=True).iloc[:, 0]) if f.exists() else None
            ok = s is not None and C.coverage_ok(s, SPEC)[0]; src, tried = ('synthetic-test' if ok else None), []
            s = s if ok else None
        else:
            s, src, tried = C.fetch_index(tk, SPEC)
        if s is None:
            manifest['indices'][tk] = dict(included=False, tried=tried); print(tk, 'EXCLUDED', tried); continue
        fn = f"{tk.replace('^', '')}.csv"; b = s.to_frame('close').to_csv().encode()
        (RAW / fn).write_bytes(b)
        manifest['indices'][tk] = dict(included=True, source=src, file=fn, sha256=C.sha256_bytes(b), n=len(s),
                                       first=str(s.index[0].date()), last=str(s.index[-1].date()), tried=tried)
        print(f'{tk:8s} {src:14s} {len(s):6d} days {s.index[0].date()} .. {s.index[-1].date()}')
    MAN.write_text(json.dumps(manifest, indent=2))
INDICES = [tk for tk, m in manifest['indices'].items() if m['included']]
if len(INDICES) < SPEC['data']['min_indices']:
    raise RuntimeError(f'only {len(INDICES)} indices passed the coverage rule (minimum {SPEC["data"]["min_indices"]})')
CLOSE = {tk: pd.read_csv(RAW / manifest['indices'][tk]['file'], index_col=0, parse_dates=True)['close'] for tk in INDICES}
print('indices included:', INDICES)

^GSPC    yahoo           19121 days 1950-01-03 .. 2025-12-31
^IXIC    yahoo           13843 days 1971-02-05 .. 2025-12-31
^GSPTSE  yahoo           11677 days 1979-06-29 .. 2025-12-31
^FTSE    yahoo           10609 days 1984-01-03 .. 2025-12-31
^GDAXI   yahoo            9609 days 1987-12-30 .. 2025-12-30
^FCHI    yahoo            9101 days 1990-03-01 .. 2025-12-31
^SSMI    yahoo            8824 days 1990-11-09 .. 2025-12-30
^N225    yahoo           14996 days 1965-01-05 .. 2025-12-30
^HSI     yahoo            9626 days 1986-12-31 .. 2025-12-31
^AXJO    yahoo            8369 days 1992-11-23 .. 2025-12-31
indices included: ['^GSPC', '^IXIC', '^GSPTSE', '^FTSE', '^GDAXI', '^FCHI', '^SSMI', '^N225', '^HSI', '^AXJO']


## 3. Phase 1: training (up to 1999), threshold calibration (2000–2004) and freezing

In [5]:
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

T_CAL0, T_CAL1 = (pd.Timestamp(x) for x in SPEC['periods']['calibration'])
T_EV0, T_EV1 = (pd.Timestamp(x) for x in SPEC['periods']['evaluation'])
EVD = SPEC['events']; WIN = SPEC['window']; BUD = SPEC['budget']
SCHED = dict(q_cap=BUD['quarter_cap'], y_cap=BUD['year_cap'], cooldown=BUD['cooldown_days'])
FEAT = {tk: C.features(CLOSE[tk]) for tk in INDICES}
VALID = {tk: FEAT[tk][C.FEATS].notna().all(axis=1).to_numpy() for tk in INDICES}
FROZEN_P = OUT / 'FROZEN.json'

if FROZEN_P.exists():
    FROZEN = json.loads(FROZEN_P.read_text())
    for k in ['M0', 'GBM']:
        if C.sha256_bytes((OUT / FROZEN['models'][k]['file']).read_bytes()) != FROZEN['models'][k]['sha256']:
            raise RuntimeError(f'GUARD FAIL: frozen {k} model changed')
    if FROZEN['spec_sha256'] != SPEC_SHA or FROZEN['raw_manifest_sha256'] != C.sha256_bytes(MAN.read_bytes()):
        raise RuntimeError('GUARD FAIL: frozen models were made under a different SPEC or raw data')
    print('phase 1 already frozen', FROZEN['frozen_utc'])
else:
    Xs, ys, ntr = [], [], {}
    for tk in INDICES:
        c = CLOSE[tk]; pre = c[c.index < T_CAL0]
        E_tr = C.catalog(pre, EVD['main_H'], EVD['theta'], EVD['refractory_days'], EVD['onset_fraction_of_theta'])
        lab, _ = C.window_labels(len(pre), E_tr, WIN['wmin'], WIN['wmax'])
        keep = VALID[tk][:len(pre)].copy(); keep[max(0, len(pre) - SPEC['periods']['training_drop_last_days']):] = False
        Xs.append(FEAT[tk].iloc[:len(pre)][C.FEATS].to_numpy(float)[keep]); ys.append((lab[keep] > 0).astype(int))
        ntr[tk] = dict(rows=int(keep.sum()), positives=int((lab[keep] > 0).sum()), events=int(len(E_tr)))
    X = np.vstack(Xs); y = np.concatenate(ys)
    fi = [C.FEATS.index('z_vol'), C.FEATS.index('ret_5')]
    m0 = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400, solver='lbfgs', random_state=17)).fit(X[:, fi], y)
    gbm = HistGradientBoostingClassifier(**SPEC['systems']['GBM']['params']).fit(X.astype(np.float32), y)
    models = {}
    for k, m in [('M0', m0), ('GBM', gbm)]:
        p = OUT / f'model_{k}.joblib'; joblib.dump(m, p); models[k] = dict(file=p.name, sha256=C.sha256_bytes(p.read_bytes()))
    thr = {}
    for tk in INDICES:
        d = FEAT[tk].index; cal = VALID[tk] & (d >= T_CAL0) & (d < T_CAL1); pos = np.flatnonzero(cal)
        Xc = FEAT[tk][C.FEATS].to_numpy(float)[cal]
        sc = {'M0': m0.predict_proba(Xc[:, fi])[:, 1], 'M1': FEAT[tk]['z_vol'].to_numpy(float)[cal], 'GBM': gbm.predict_proba(Xc.astype(np.float32))[:, 1]}
        thr[tk] = {k: dict(zip(['threshold', 'cal_alerts'], C.calibrate(pos, d[cal], v, SCHED))) for k, v in sc.items()}
    FROZEN = dict(frozen_utc=pd.Timestamp.now(tz='UTC').isoformat(), spec_sha256=SPEC_SHA, raw_manifest_sha256=C.sha256_bytes(MAN.read_bytes()),
                  models=models, thresholds=thr, training=ntr, training_rows=int(len(y)), training_positive_share=float(y.mean()))
    FROZEN_P.write_text(json.dumps(FROZEN, indent=2))
    print(f'phase 1 frozen: {len(y):,} training days ({y.mean():.1%} positive)')
FROZEN_SHA = C.sha256_bytes(FROZEN_P.read_bytes())
M0 = joblib.load(OUT / FROZEN['models']['M0']['file']); GBM = joblib.load(OUT / FROZEN['models']['GBM']['file'])
display(pd.DataFrame({tk: {k: v['cal_alerts'] for k, v in FROZEN['thresholds'][tk].items()} for tk in INDICES}).T.rename(columns=lambda c: f'{c} calibration alerts'))

phase 1 frozen: 47,743 training days (6.5% positive)


,M0 calibration alerts,M1 calibration alerts,GBM calibration alerts
^GSPC,20,20,20
^IXIC,20,20,20
^GSPTSE,20,20,20
^FTSE,20,20,20
^GDAXI,20,20,20
^FCHI,20,20,20
^SSMI,20,20,20
^N225,20,20,20
^HSI,20,20,20
^AXJO,20,20,20


## 4. Phase 2: first computation of the evaluation results

In [6]:
LOCK = OUT / 'EVALUATED.lock'
if LOCK.exists():
    raise RuntimeError(f'already evaluated (one-shot): {json.loads(LOCK.read_text())}')
log = json.loads(RUN_LOG.read_text()); log.append(dict(utc=pd.Timestamp.now(tz='UTC').isoformat(), event='phase 2 started', frozen_sha256=FROZEN_SHA))
RUN_LOG.write_text(json.dumps(log, indent=2))
T0 = time.time()
SYS_ALERT = ['M0', 'M1', 'GBM']; SYS_ALL = ['M0', 'M1', 'GBM', 'B0', 'B1']
fi = [C.FEATS.index('z_vol'), C.FEATS.index('ret_5')]
QBASE = T_EV0.year * 4 + (T_EV0.month - 1) // 3
NQ = (T_EV1.year * 4 + (T_EV1.month - 1) // 3) - QBASE

D = {}
for tk in INDICES:
    c = CLOSE[tk]; d = c.index; n = len(c)
    ev = VALID[tk] & (d >= T_EV0) & (d < T_EV1); evpos = np.flatnonzero(ev)
    Xe = FEAT[tk][C.FEATS].to_numpy(float)
    S = {k: np.full(n, np.nan) for k in SYS_ALL}
    S['M0'][ev] = M0.predict_proba(Xe[ev][:, fi])[:, 1]; S['GBM'][ev] = GBM.predict_proba(Xe[ev].astype(np.float32))[:, 1]
    S['M1'][ev] = FEAT[tk]['z_vol'].to_numpy(float)[ev]; S['B0'][ev] = FEAT[tk]['B0'].to_numpy(float)[ev]; S['B1'][ev] = FEAT[tk]['B1'].to_numpy(float)[ev]
    A = {k: C.schedule(evpos, d[ev], S[k][ev] >= FROZEN['thresholds'][tk][k]['threshold'], **SCHED) for k in SYS_ALERT}
    cats = {}
    for H in [EVD['main_H']] + EVD['secondary_H']:
        E = C.catalog(c, H, EVD['theta'], EVD['refractory_days'], EVD['onset_fraction_of_theta'])
        E = E[(E.breach_pos >= evpos[0] + SPEC['periods']['event_must_breach_after_eval_start_days']) & (E.breach_date < T_EV1)].reset_index(drop=True)
        cats[H] = E
    quarter = np.full(n, -1, np.int64); dq = d.year.to_numpy() * 4 + (d.month.to_numpy() - 1) // 3 - QBASE; quarter[ev] = dq[ev]
    rvdec = np.full(n, -1, np.int64); rvdec[ev] = np.minimum((pd.Series(S['B0'][ev]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    D[tk] = dict(n=n, dates=d, ev=ev, evpos=evpos, S=S, A=A, E=cats, quarter=quarter, rvdec=rvdec)
print('alerts:', {k: sum(len(D[tk]['A'][k]) for tk in INDICES) for k in SYS_ALERT},
      '| events:', {H: sum(len(D[tk]['E'][H]) for tk in INDICES) for H in D[INDICES[0]]['E']})

# ---- event and alert lists (dates and positions; no index levels)
rows = []
for tk in INDICES:
    for H, E in D[tk]['E'].items():
        e = E.copy(); e.insert(0, 'H', H); e.insert(0, 'index', tk); rows.append(e)
pd.concat(rows, ignore_index=True).to_csv(OUT / '1_events.csv', index=False)
pd.DataFrame([dict(index=tk, system=k, alert_date=D[tk]['dates'][p].date(), alert_pos=int(p)) for tk in INDICES for k in SYS_ALERT for p in D[tk]['A'][k]]).to_csv(OUT / '1b_alerts.csv', index=False)

# ---- E2 feasibility (labels only)
rows = []
for H in [10, 20, 60]:
    for wm in SPEC['feasibility']['wmax']:
        L = np.concatenate([C.feasibility(D[tk]['E'][H], wm, WIN['wmin']) for tk in INDICES]) if H in D[INDICES[0]]['E'] else np.array([])
        rows.append(dict(H=H, wmax=wm, events=len(L), impossible=int((L == 0).sum()), F=float((L > 0).mean()) if len(L) else np.nan))
T2 = pd.DataFrame(rows); T2.to_csv(OUT / '2_feasibility.csv', index=False)
bad = T2[(T2.H <= T2.wmax) & (T2.F < 1)]
if len(bad):
    raise RuntimeError(f'GUARD FAIL (Corollary 1): {bad.to_dict("records")}')
display(T2.pivot(index='H', columns='wmax', values='F').round(3))

alerts: {'M0': 805, 'M1': 823, 'GBM': 713} | events: {20: 241, 10: 146, 60: 233}


wmax,5,10,20,40,60
H,,,,,
10,0.315,1.000,1.000,1.000,1.0
20,0.120,0.398,1.000,1.000,1.0
60,0.047,0.112,0.386,0.747,1.0


In [7]:
# ---- E1/E3 per-day AUCs with a common calendar-quarter bootstrap
rng = np.random.default_rng(SPEC['per_day']['bootstrap']['seed'])
NB = SPEC['per_day']['bootstrap']['draws']
WQ = np.vstack([np.ones(NQ)] + [np.bincount(rng.integers(0, NQ, NQ), minlength=NQ).astype(float) for _ in range(NB)])   # row 0 = point estimate


def preps(H, negset='main', with_reg=True):
    """Per index and system: AUC preparations for standard, pre-onset, in-progress and within-regime pre-onset."""
    P = {}
    for tk in INDICES:
        x = D[tk]; lab, _ = C.window_labels(x['n'], x['E'][H], WIN['wmin'], WIN['wmax'])
        keep = x['ev'] & np.all([np.isfinite(x['S'][k]) for k in SYS_ALL], axis=0)
        neg = keep & (lab == 0)
        if negset == 'secondary':
            for b in x['E'][H].breach_pos:
                neg[b:b + WIN['wmax'] + 1] = False
        P[tk] = {}
        for k in SYS_ALL:
            s = x['S'][k]
            P[tk][k] = dict(std=C.AucPrep(s, lab > 0, neg | (keep & (lab > 0)), x['quarter']),
                            pre=C.AucPrep(s, lab == 1, neg | (keep & (lab == 1)), x['quarter']),
                            inp=C.AucPrep(s, lab == 2, neg | (keep & (lab == 2)), x['quarter']),
                            reg=C.StratAucPrep(s, lab == 1, neg | (keep & (lab == 1)), x['quarter'], x['rvdec']) if with_reg else None,
                            q_in=x['quarter'][keep & (lab == 2)], q_pos=x['quarter'][keep & (lab > 0)])
    return P


def evaluate(P, w):
    """Equal-weighted averages over indices for one quarter-weight vector w."""
    out = {}
    for k in SYS_ALL:
        a = {m: [] for m in ['std', 'pre', 'inp', 'reg']}; num, den, pis = [], [], []
        for tk in INDICES:
            p = P[tk][k]; v = {m: (p[m].auc(w) if p[m] is not None else np.nan) for m in ['std', 'pre', 'inp', 'reg']}
            wpos = w[p['q_pos']].sum(); pi = w[p['q_in']].sum() / wpos if wpos > 0 else np.nan
            for m in a:
                a[m].append(v[m])
            if np.isfinite(v['std']) and np.isfinite(v['inp']) and np.isfinite(pi):
                num.append(pi * (v['inp'] - 0.5)); den.append(v['std'] - 0.5); pis.append(pi)
        r = {m: (float(np.nanmean(a[m])) if np.isfinite(a[m]).any() else np.nan) for m in a}
        r['pi'] = float(np.mean(pis)) if pis else np.nan; r['excess'] = float(np.mean(den)) if den else np.nan
        r['S_in'] = float(np.mean(num) / np.mean(den)) if den and np.mean(den) != 0 else np.nan
        out[k] = r
    return out


BOOT = {}
for H, negset in [(20, 'main'), (60, 'main'), (10, 'main'), (20, 'secondary'), (60, 'secondary')]:
    t_ = time.time(); P = preps(H, negset, with_reg=(H == 20 and negset == 'main'))
    BOOT[(H, negset)] = [evaluate(P, WQ[i]) for i in range(NB + 1)]
    print(f'per-day bootstrap H = {H}, negatives {negset}: {time.time() - t_:.0f}s')

rows = []
for (H, negset), draws in BOOT.items():
    for k in SYS_ALL:
        for m in ['std', 'pre', 'inp', 'reg', 'pi', 'excess', 'S_in']:
            v = np.array([d_[k][m] for d_ in draws[1:]])
            rows.append(dict(H=H, negatives=negset, system=k, measure=m, point=draws[0][k][m],
                             lo95=float(np.nanpercentile(v, 2.5)), hi95=float(np.nanpercentile(v, 97.5)), finite_draws=int(np.isfinite(v).sum())))
    for m in ['std', 'pre', 'inp', 'reg']:
        v = np.array([d_['GBM'][m] - d_['M0'][m] for d_ in draws[1:]])
        rows.append(dict(H=H, negatives=negset, system='GBM - M0', measure=m, point=draws[0]['GBM'][m] - draws[0]['M0'][m],
                         lo95=float(np.nanpercentile(v, 2.5)), hi95=float(np.nanpercentile(v, 97.5)), finite_draws=int(np.isfinite(v).sum())))
T1 = pd.DataFrame(rows); T1.to_csv(OUT / '3_per_day_auc.csv', index=False)
display(T1[(T1.H == 20) & (T1.negatives == 'main') & T1.measure.isin(['std', 'pre', 'inp', 'reg', 'pi', 'S_in'])].pivot(index='system', columns='measure', values='point').round(3))

# per-index point values (descriptive)
P20 = preps(20); rows = []
for tk in INDICES:
    for k in SYS_ALL:
        p = P20[tk][k]; w = np.ones(NQ)
        rows.append(dict(index=tk, system=k, events=len(D[tk]['E'][20]), std=p['std'].auc(w), pre=p['pre'].auc(w), inp=p['inp'].auc(w), reg=p['reg'].auc(w),
                         pi=len(p['q_in']) / max(len(p['q_pos']), 1)))
pd.DataFrame(rows).to_csv(OUT / '3b_per_day_auc_by_index.csv', index=False)

per-day bootstrap H = 20, negatives main: 78s
per-day bootstrap H = 60, negatives main: 41s
per-day bootstrap H = 10, negatives main: 40s
per-day bootstrap H = 20, negatives secondary: 36s
per-day bootstrap H = 60, negatives secondary: 38s


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1396: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


measure,S_in,inp,pi,pre,reg,std
system,,,,,,
B0,0.629,0.770,0.537,0.684,0.567,0.730
B1,0.843,0.873,0.537,0.579,0.503,0.737
GBM,0.756,0.841,0.537,0.626,0.534,0.742
GBM - M0,NaN,0.018,NaN,0.038,0.037,0.027
M0,0.807,0.824,0.537,0.589,0.497,0.715
M1,0.589,0.700,0.537,0.662,0.541,0.683


In [8]:
# ---- E4 event level (D20): observed catches, three nulls, market episodes, episode bootstrap
H = EVD['main_H']
EV = pd.concat([D[tk]['E'][H].assign(index=tk) for tk in INDICES], ignore_index=True)
SL = {tk: np.flatnonzero(EV['index'].to_numpy() == tk) for tk in INDICES}
G_MAIN = SPEC['event_level']['episodes']['link_gap_calendar_days']; G_SEC = SPEC['event_level']['episodes']['secondary_gap_calendar_days']
EPI = C.episodes(EV.breach_date, G_MAIN); EPI2 = C.episodes(EV.breach_date, G_SEC); NEP = int(EPI.max()) + 1
EV['episode'] = EPI; EV['episode_secondary'] = EPI2
EV['episode_indices'] = EV.groupby('episode')['index'].transform('nunique')


def catches(alerts_by_index):
    pre = np.zeros(len(EV), bool); hit = np.zeros(len(EV), bool); al = np.full(len(EV), -1, np.int64)
    for tk in INDICES:
        h, p, a = C.match(D[tk]['E'][H], alerts_by_index[tk], WIN['wmin'], WIN['wmax'])
        hit[SL[tk]] = h; pre[SL[tk]] = p; al[SL[tk]] = a
    return hit, pre, al


OBS = {}
for k in SYS_ALERT:
    hit, pre, al = catches({tk: D[tk]['A'][k] for tk in INDICES})
    OBS[k] = dict(hit=hit, pre=pre, inp=hit & ~pre, alert=al, alerts=sum(len(D[tk]['A'][k]) for tk in INDICES))
    EV[f'{k}_pre'] = pre.astype(int); EV[f'{k}_in'] = (hit & ~pre).astype(int)
    EV[f'{k}_alert_minus_onset_days'] = np.where(hit, al - EV.onset_pos.to_numpy(), np.nan)

span = (T_EV1 - T_EV0).days
QPOS = {tk: C.quarter_positions(D[tk]['evpos'], D[tk]['quarter']) for tk in INDICES}
rngN = np.random.default_rng(SPEC['event_level']['seed_nulls'])
NULLS = {'budget-preserving': SPEC['event_level']['nulls']['budget_preserving_regime']['draws'],
         'joint circular': SPEC['event_level']['nulls']['joint_circular']['draws'],
         'short joint shift': SPEC['event_level']['nulls']['short_joint_shift']['draws']}
NR = {}; NSTAT = {k: {} for k in SYS_ALERT}
for k in SYS_ALERT:
    for nname, n_ in NULLS.items():
        t_ = time.time(); p_pre = np.zeros(len(EV)); p_in = np.zeros(len(EV))
        cnt = {m: np.empty(n_, np.int64) for m in ['pre', 'inp', 'ep_pre', 'ep_inp']}
        for i in range(n_):
            if nname == 'budget-preserving':
                sh = {tk: C.null_budget_preserving(rngN, D[tk]['A'][k], QPOS[tk], D[tk]['quarter'], D[tk]['rvdec'], BUD['cooldown_days'], NSTAT[k]) for tk in INDICES}
            elif nname == 'joint circular':
                sh = C.null_shift({tk: D[tk]['A'][k] for tk in INDICES}, {tk: D[tk]['dates'] for tk in INDICES}, SPEC['periods']['evaluation'], int(rngN.integers(365, span - 365)))
            else:
                sh = C.null_shift({tk: D[tk]['A'][k] for tk in INDICES}, {tk: D[tk]['dates'] for tk in INDICES}, SPEC['periods']['evaluation'],
                                  int(rngN.integers(28, 85)) * int(rngN.choice([-1, 1])))
            hit, pre, _ = catches(sh); inp = hit & ~pre
            p_pre += pre; p_in += inp
            cnt['pre'][i] = pre.sum(); cnt['inp'][i] = inp.sum(); cnt['ep_pre'][i] = np.unique(EPI[pre]).size; cnt['ep_inp'][i] = np.unique(EPI[inp]).size
        NR[(k, nname)] = dict(n=n_, cnt=cnt, p_pre=p_pre / n_, p_in=p_in / n_)
        print(f'{k:>3} {nname:<18} {n_:>5} draws ({time.time() - t_:.0f}s)')

rows = []
for k in SYS_ALERT:
    for nname in NULLS:
        x = NR[(k, nname)]
        for m, obs in [('pre', OBS[k]['pre'].sum()), ('inp', OBS[k]['inp'].sum()),
                       ('ep_pre', np.unique(EPI[OBS[k]['pre']]).size), ('ep_inp', np.unique(EPI[OBS[k]['inp']]).size)]:
            v = x['cnt'][m]
            rows.append(dict(system=k, null=nname, statistic=m, observed=int(obs), null_mean=float(v.mean()), null_q05=float(np.percentile(v, 5)),
                             null_q95=float(np.percentile(v, 95)), p_one_sided=(1 + int((v >= obs).sum())) / (len(v) + 1), draws=len(v),
                             events=len(EV), episodes=NEP, alerts=OBS[k]['alerts']))
T4 = pd.DataFrame(rows)
for m in ['ep_pre', 'ep_inp']:
    sel = (T4['null'] == 'budget-preserving') & (T4.statistic == m)
    T4.loc[sel, 'p_holm'] = C.holm(T4.loc[sel, 'p_one_sided'].to_numpy())
T4.to_csv(OUT / '4_event_level_tests.csv', index=False)
pd.DataFrame(NSTAT).T.fillna(0).astype(int).to_csv(OUT / '4b_null_placement_diagnostics.csv')
print('budget-preserving placements:', NSTAT)
display(T4[T4['null'] == 'budget-preserving'].round(4))

rngB = np.random.default_rng(SPEC['event_level']['episode_bootstrap']['seed']); NBE = SPEC['event_level']['episode_bootstrap']['draws']
IDX = rngB.integers(0, NEP, size=(NBE, NEP))
rows = []; DIFF = {}
for k in SYS_ALERT:
    for nname in NULLS:
        x = NR[(k, nname)]; al = OBS[k]['alerts']
        ex_pre = np.bincount(EPI, OBS[k]['pre'] - x['p_pre'], minlength=NEP); ex_in = np.bincount(EPI, OBS[k]['inp'] - x['p_in'], minlength=NEP)
        b_pre = 100 * ex_pre[IDX].sum(1) / al; b_in = 100 * ex_in[IDX].sum(1) / al; b_d = b_in - b_pre
        for m, pt, b in [('pre', 100 * ex_pre.sum() / al, b_pre), ('inp', 100 * ex_in.sum() / al, b_in), ('inp - pre', 100 * (ex_in.sum() - ex_pre.sum()) / al, b_d)]:
            rows.append(dict(system=k, null=nname, measure=m, excess_per_100_alerts=float(pt), lo95=float(np.percentile(b, 2.5)), hi95=float(np.percentile(b, 97.5)),
                             alerts=al, episodes=NEP, draws=NBE))
        if nname == 'budget-preserving':
            DIFF[k] = (float(np.percentile(b_d, 2.5)), float(np.percentile(b_d, 97.5)))
T5 = pd.DataFrame(rows); T5.to_csv(OUT / '5_effect_sizes.csv', index=False)
display(T5[T5['null'] == 'budget-preserving'].round(3))

size = EV['episode_indices'].to_numpy(); rows = []
for k in SYS_ALERT:
    x = NR[(k, 'budget-preserving')]
    for lo_, hi_, lab_ in [(1, 1, '1 index'), (2, 4, '2-4 indices'), (5, 99, '5+ indices'), (1, 99, 'all')]:
        sel = (size >= lo_) & (size <= hi_)
        for m, o, p in [('pre', OBS[k]['pre'], x['p_pre']), ('inp', OBS[k]['inp'], x['p_in'])]:
            rows.append(dict(system=k, endpoint=m, episode_size=lab_, events=int(sel.sum()), observed=int(o[sel].sum()), expected=float(p[sel].sum())))
pd.DataFrame(rows).to_csv(OUT / '5b_excess_by_episode_size.csv', index=False)

rows = []
for k in SYS_ALERT:
    for m in ['pre', 'inp']:
        v = EV.loc[OBS[k][m].astype(bool), f'{k}_alert_minus_onset_days'].to_numpy()
        rows.append(dict(system=k, endpoint=m, n=len(v), **({f'alert_minus_onset_p{q}': float(np.percentile(v, q)) for q in (10, 25, 50, 75, 90)} if len(v) else {})))
pd.DataFrame(rows).to_csv(OUT / '6_lead_times.csv', index=False)
EV.drop(columns=[]).to_csv(OUT / '1c_event_level.csv', index=False)

 M0 budget-preserving   5000 draws (170s)
 M0 joint circular      5000 draws (36s)
 M0 short joint shift   1000 draws (7s)
 M1 budget-preserving   5000 draws (176s)
 M1 joint circular      5000 draws (35s)
 M1 short joint shift   1000 draws (8s)
GBM budget-preserving   5000 draws (149s)
GBM joint circular      5000 draws (35s)
GBM short joint shift   1000 draws (7s)
budget-preserving placements: {'M0': {'own decile': 2579134, 'other decile': 1445244, 'kept original': 622}, 'M1': {'other decile': 1412312, 'own decile': 2702688}, 'GBM': {'own decile': 2151967, 'other decile': 1412697, 'kept original': 336}}


,system,null,statistic,observed,null_mean,null_q05,null_q95,p_one_sided,draws,events,episodes,alerts,p_holm
0,M0,budget-preserving,pre,21,45.7768,40.0,51.0,1.0000,5000,241,52,805,NaN
1,M0,budget-preserving,inp,78,62.1400,56.0,68.0,0.0004,5000,241,52,805,NaN
2,M0,budget-preserving,ep_pre,9,20.8454,18.0,23.0,1.0000,5000,241,52,805,1.0000
3,M0,budget-preserving,ep_inp,28,30.4678,27.0,33.0,0.9490,5000,241,52,805,1.0000
12,M1,budget-preserving,pre,32,49.3702,44.0,55.0,1.0000,5000,241,52,823,NaN
13,M1,budget-preserving,inp,49,62.0384,56.0,68.0,1.0000,5000,241,52,823,NaN
14,M1,budget-preserving,ep_pre,13,20.4778,18.0,23.0,1.0000,5000,241,52,823,1.0000
15,M1,budget-preserving,ep_inp,20,27.9290,25.0,31.0,1.0000,5000,241,52,823,1.0000
24,GBM,budget-preserving,pre,30,31.2228,26.0,37.0,0.7031,5000,241,52,713,NaN
25,GBM,budget-preserving,inp,85,53.3378,48.0,59.0,0.0002,5000,241,52,713,NaN


,system,null,measure,excess_per_100_alerts,lo95,hi95,alerts,episodes,draws
0,M0,budget-preserving,pre,-3.078,-6.346,0.178,805,52,10000
1,M0,budget-preserving,inp,1.970,-2.826,6.959,805,52,10000
2,M0,budget-preserving,inp - pre,5.048,-0.582,11.412,805,52,10000
9,M1,budget-preserving,pre,-2.111,-5.555,1.293,823,52,10000
10,M1,budget-preserving,inp,-1.584,-5.402,2.595,823,52,10000
11,M1,budget-preserving,inp - pre,0.526,-4.561,6.209,823,52,10000
18,GBM,budget-preserving,pre,-0.172,-3.019,2.822,713,52,10000
19,GBM,budget-preserving,inp,4.441,-0.268,9.479,713,52,10000
20,GBM,budget-preserving,inp - pre,4.612,-0.952,10.659,713,52,10000


In [9]:
# ---- readings fixed in advance
R = T1.set_index(['H', 'negatives', 'system', 'measure'])
draws20 = BOOT[(20, 'main')][1:]
den_small = float(np.mean([d_['M0']['excess'] <= 0.02 for d_ in draws20]))
s_lo, s_hi = R.loc[(20, 'main', 'M0', 'S_in'), ['lo95', 'hi95']]
if den_small > 0.025:
    q1 = 'inconclusive (not evaluable: above-chance AUC <= 0.02 in %.1f%% of draws)' % (100 * den_small)
else:
    q1 = 'supported' if s_lo > 0.25 else ('contradicted' if s_hi < 0.25 else 'inconclusive')
h1 = {k: bool(T4[(T4.system == k) & (T4['null'] == 'budget-preserving') & (T4.statistic == 'ep_inp')].p_holm.iloc[0] < 0.05) for k in SYS_ALERT}
h2 = {k: bool(T4[(T4.system == k) & (T4['null'] == 'budget-preserving') & (T4.statistic == 'ep_pre')].p_holm.iloc[0] < 0.05) for k in SYS_ALERT}
held = [k for k in SYS_ALERT if h1[k]]
if len(held) >= 2 and all(DIFF[k][0] > 0 for k in held):
    q2 = 'supported'
elif sum(DIFF[k][1] < 0 for k in SYS_ALERT) >= 2:
    q2 = 'contradicted'
else:
    q2 = 'inconclusive'
F60 = float(T2[(T2.H == 60) & (T2.wmax == 20)].F.iloc[0])
d_std = R.loc[(60, 'main', 'GBM - M0', 'std')]; d_pre = R.loc[(60, 'main', 'GBM - M0', 'pre')]
ex = lambda r: 1 if r.lo95 > 0 else (-1 if r.hi95 < 0 else 0)
q4 = 'supported' if ex(d_std) * ex(d_pre) == -1 else ('contradicted' if ex(d_std) * ex(d_pre) == 1 else 'inconclusive')
READ = {'Q1': dict(reading=q1, S_in=R.loc[(20, 'main', 'M0', 'S_in'), 'point'], lo95=s_lo, hi95=s_hi, above_half=bool(R.loc[(20, 'main', 'M0', 'S_in'), 'point'] > 0.5)),
        'Q2': dict(reading=q2, H1_in_progress=h1, H2_pre_onset=h2, diff_in_minus_pre_95=DIFF),
        'Q3': dict(reading='F < 1' if F60 < 1 else 'F = 1', F_D60_wmax20=F60, F_D10=float(T2[(T2.H == 10) & (T2.wmax == 20)].F.iloc[0]), F_D20=float(T2[(T2.H == 20) & (T2.wmax == 20)].F.iloc[0])),
        'Q4': dict(reading=q4, std=d_std[['point', 'lo95', 'hi95']].to_dict(), pre=d_pre[['point', 'lo95', 'hi95']].to_dict())}
SUMMARY = dict(spec_sha256=SPEC_SHA, frozen_sha256=FROZEN_SHA, registration_doi=REGISTRATION_DOI, indices=INDICES,
               excluded=[tk for tk, m in manifest['indices'].items() if not m['included']], sources={tk: manifest['indices'][tk]['source'] for tk in INDICES},
               events={str(H_): int(sum(len(D[tk]['E'][H_]) for tk in INDICES)) for H_ in D[INDICES[0]]['E']}, episodes=NEP,
               alerts={k: OBS[k]['alerts'] for k in SYS_ALERT}, readings=READ, runtime_seconds=round(time.time() - T0, 1))
(OUT / '38_SUMMARY.json').write_text(json.dumps(SUMMARY, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
LOCK.write_text(json.dumps(dict(evaluated_utc=pd.Timestamp.now(tz='UTC').isoformat(), spec_sha256=SPEC_SHA, frozen_sha256=FROZEN_SHA)))
log = json.loads(RUN_LOG.read_text()); log.append(dict(utc=pd.Timestamp.now(tz='UTC').isoformat(), event='phase 2 finished; locked')); RUN_LOG.write_text(json.dumps(log, indent=2))
print(json.dumps(READ, indent=2, default=str))
print(f'LOCKED. done in {time.time() - T0:.0f}s ->', OUT)

{
  "Q1": {
    "reading": "supported",
    "S_in": 0.8074257904492735,
    "lo95": 0.6896038495201482,
    "hi95": 0.9525747160580819,
    "above_half": true
  },
  "Q2": {
    "reading": "inconclusive",
    "H1_in_progress": {
      "M0": false,
      "M1": false,
      "GBM": true
    },
    "H2_pre_onset": {
      "M0": false,
      "M1": false,
      "GBM": false
    },
    "diff_in_minus_pre_95": {
      "M0": [
        -0.5823844720496891,
        11.4123
      ],
      "M1": [
        -4.561106925880923,
        6.209188942891856
      ],
      "GBM": [
        -0.9519747545582044,
        10.658920757363255
      ]
    }
  },
  "Q3": {
    "reading": "F < 1",
    "F_D60_wmax20": 0.38626609442060084,
    "F_D10": 1.0,
    "F_D20": 1.0
  },
  "Q4": {
    "reading": "inconclusive",
    "std": {
      "point": 0.027026260964644555,
      "lo95": -0.018023922392079374,
      "hi95": 0.07458955319744032
    },
    "pre": {
      "point": 0.0058156046975536,
      "lo95": -0.08971959